# FEM validation checks — design document and walkthrough

This notebook is the **design document** for the `fem_checks` package, and it **runs**. Every
step of the tool is shown in the order the tool performs it: a markdown section explains *what*
the step does and *why*, then a code cell calls the package's own functions so you can see the
intermediate results on a real model.

**Who this is for**

| Reader | Read |
|---|---|
| Analyst running the checks | §0 and §10 (how to run, how to read the report) |
| Reviewer of a model check-out | §1 (what is checked and the pass/fail criteria) and each check's section |
| Developer changing the tool | everything, plus the module docstrings in `fem_checks/*.py` |

**Source documents in this repo**

* `FEMCI-The-Book.pdf` — NASA GSFC FEMCI "The Book", section *Validity Checks*. References look
  like `femci ~2082` = line in `pdftotext -layout FEMCI-The-Book.pdf femci.txt`.
* `QRG.pdf` — Simcenter Nastran Quick Reference Guide (2025). References look like `qrg ~18934`.
* `research-fem-checks.md` / `plan-fem-checks.md` — the research and the approved plan, including
  every decision and why it was made.

> **Status:** implemented and tested against the sample Simcenter outputs and the full model deck
> in this repo. Not yet exercised against a live Nastran run from this tool — see the first-run
> checklist at the end of `plan-fem-checks.md`.

## 0. How the tool is used

```
pip install -r fem_checks/requirements.txt
python -m fem_checks                          # GUI
python -m fem_checks <out>/fem_checks.json    # repeat a run without the GUI
python -m fem_checks <out>/fem_checks.json --no-run   # re-evaluate outputs already on disk
```

The user supplies a FEMAP-exported deck (`.dat`) and the path to their `nastran.exe`. The tool:

1. reads the model, summarizes it and picks a kinematic-mount grid,
2. writes the model's bulk data once (plus a uniform-CTE copy for the thermal soak),
3. writes five check decks and runs Nastran on each,
4. parses the `.f06` and `.op2` files,
5. evaluates PASS / FAIL / INFO / ERROR per check, and
6. writes `fem_checks_report.pdf` and `results.json` into one output folder.

This notebook does exactly that, one step per section, on the sample model `BD IRAD V2`.

## 1. What is checked, and the pass/fail criteria

FEMCI's four *mathematical validity checks* (femci ~1880) plus two the user added. The criteria
column is what the code enforces; the decision numbers refer to `plan-fem-checks.md`.

| # | Check | Run | Constraint | PASS when |
|---|---|---|---|---|
| 01 | Free-free modes + GROUNDCHECK | SOL 103 | none | $\max(f_1..f_6)/f_7 \le 10^{-4}$ (decision #1) **and** every GROUNDCHECK set prints PASS |
| 02 | Constrained modes | SOL 103 | model's real SPC set | report only (INFO): gimbals can have near-zero bearing modes |
| 03 | Unit enforced displacement / rotation | SOL 101, 6 subcases | one grid 123456 + SPCD | mount reaction ≤ 0.1 lbf / 1 in-lbf; displacement field = rigid-body motion to $10^{-3}$; element centroid forces ≤ 0.1 / 1.0 |
| 04 | Unit gravity X/Y/Z | SOL 101, 3 subcases | kinematic mount (or real set) | OLOAD = −SPCFORCE and OLOAD = $m g$, each to $10^{-4}$ |
| 05 | Thermal soak +1° | SOL 101 | kinematic mount | displacement = $\alpha\,\Delta T\,(\mathbf r-\mathbf r_m)$ to $10^{-3}$; $\sigma_{vm,max}/(E\alpha\Delta T)\le10^{-3}$ |
| — | Mass, CG, inertia | from 01 | — | each value the user enters is within its tolerance |
| — | Model summary | from the deck | — | INFO: counts, element quality vs Simcenter's GEOMCHECK defaults |

Notable departures from the Book, each decided with the user:

* **No absolute rigid-body frequency limit.** FEMCI's ≤ 1e-4 Hz (femci ~2095) fails 4 of 6
  rigid-body modes on this model even though GROUNDCHECK passes by orders of magnitude. The ratio
  to the first elastic mode is what shows grounding.
* **Rigid elements expand in the thermal soak.** Every RBAR/RBE1/RBE2/RBE3/RROD gets the uniform
  CTE and the deck runs `RIGID=LAGRAN` (qrg ~26012), matching how the user's production thermal
  runs are set up.
* **Shell element forces are judged at the centroid**, not the CORNER (bilinear) points — see §8.3.

## 2. Setup

Paths and settings. `Settings` is the same object the GUI fills and `fem_checks.json` stores.
`DEMO = True` means *don't run Nastran*: instead the user's real sample outputs in the repo root are
copied in under the check-deck names, so every later step has real Simcenter output to work on.
Set `DEMO = False` and point `NASTRAN` at `nastran.exe` to run the real thing.

In [1]:
import shutil, sys, tempfile
from pathlib import Path

# The notebook lives in fem_checks/; the repo root holds the package and the sample files.
REPO = Path.cwd().parent if Path.cwd().name == "fem_checks" else Path.cwd()
sys.path.insert(0, str(REPO))

from fem_checks.settings import Settings, UNIT_SYSTEMS
from fem_checks import model as M, decks as D, results as R, checks as C, report as RP
from fem_checks.f06 import parse_f06
from fem_checks.runner import run_nastran, RunResult

DEMO = True                      # True: reuse the sample outputs; False: run Nastran
NASTRAN = r"C:\Program Files\Siemens\FEMAP 2512\NASTRAN\bin\nastran.exe"   # used when DEMO = False

s = Settings(
    deck=str(REPO / "1G_unit_load_check-0000.dat"),   # full FEMAP-exported model
    nastran_exe=NASTRAN,
    # Outside the repo: the demo copies ~30 MB of sample OP2s and writes decks/bulk data.
    output_dir=str(Path(tempfile.gettempdir()) / "fem_checks_demo"),
    units="in-lbf-s",            # g = 386.0886 in/s^2; limits 0.1 lbf / 1.0 in-lbf
    mount_grid=199999,           # the sample runs used this grid; 0 would auto-pick (see §5)
    real_spc_set=3,              # the model's real boundary-condition set
)
s.checks["thermal"] = True       # thermal is off by default (only for thermal-distortion models)
s.targets.mass = 1.9             # optional CAD target, model mass units (lbf-s^2/in)
out = s.resolved_output_dir(); out.mkdir(exist_ok=True)
print("problems:", s.validate(need_nastran=not DEMO) or "none")
print("output folder:", out)

problems: none
output folder: /tmp/fem_checks_demo


## 3. Read the model

`model.read_model` reads the deck with pyNastran in Simcenter mode (`mode='nx'`).

* **FEMAP checksum.** FEMAP writes a checksum after `ENDDATA` (`ENDDATA b482e8c1`). The tool strips
  it into a temporary copy before reading. The decks the tool writes carry no checksum; the user
  confirmed Simcenter runs them.
* **Model summary** (`model.model_info`): card and element counts, bounding box, pyNastran mass
  properties (used only to pick the mount — Nastran's own GPWG is the authoritative mass), the
  largest Young's modulus (the thermal stress scale), and element quality.

In [2]:
model = M.read_model(Path(s.deck), out / "work")
info = M.model_info(model, Path(s.deck), log=print)
print(f"{info.n_grids} grids, {info.n_elements} elements, {info.n_rigid} rigid, {info.n_masses} masses")
print("element counts:", info.element_counts)
print(f"pyNastran mass {info.mass:.6g}, CG {[round(c, 4) for c in info.cg]}, size L = {info.size:.4g}")
print("E_max =", info.e_max, "  model PARAMs:", info.params)

/usr/local/lib/python3.13/site-packages/pyNastran/bdf/cards/elements/mass.py:1393: UserWarning: CONM2 (eid=100002) inertia method for CID != 0 is not implemented
  warnings.warn(f'CONM2 (eid={self.eid}) inertia method for CID != 0 is not implemented')
/usr/local/lib/python3.13/site-packages/pyNastran/bdf/cards/elements/mass.py:1393: UserWarning: CONM2 (eid=100003) inertia method for CID != 0 is not implemented
  warnings.warn(f'CONM2 (eid={self.eid}) inertia method for CID != 0 is not implemented')
/usr/local/lib/python3.13/site-packages/pyNastran/bdf/cards/elements/mass.py:1393: UserWarning: CONM2 (eid=700001) inertia method for CID != 0 is not implemented
  warnings.warn(f'CONM2 (eid={self.eid}) inertia method for CID != 0 is not implemented')
/usr/local/lib/python3.13/site-packages/pyNastran/bdf/cards/elements/mass.py:1393: UserWarning: CONM2 (eid=700005) inertia method for CID != 0 is not implemented
  warnings.warn(f'CONM2 (eid={self.eid}) inertia method for CID != 0 is not implem

  map_elements = 5000


  map_elements = 10000


  map_elements = 15000


  map_elements = 5000


  map_elements = 10000


  map_elements = 15000


24851 grids, 16632 elements, 18 rigid, 7 masses
element counts: {'CBEAM': 17, 'CBUSH': 3, 'CHEXA': 8856, 'CONM2': 7, 'CPENTA': 128, 'CQUAD4': 7620, 'CTRIA3': 8, 'RBE2': 16, 'RBE3': 2}
pyNastran mass 1.90061, CG [0.396, -4.4358, 0.0], size L = 69.95
E_max = 29000000.0   model PARAMs: {'PRGPST': 'NO', 'POST': -1, 'OGEOM': 'NO', 'AUTOSPC': 'YES', 'K6ROT': 100.0, 'GRDPNT': 0}


### 3.1 Traceability

Every check deck must say which model it came from (user request). `model.read_trace` pulls:

* `TITLE` / `SUBTITLE` from the FEMAP case control — the check decks keep them as their own
  TITLE/SUBTITLE so every `.f06` page names the model; the check name goes in `LABEL`;
* FEMAP's `$ From Model :` / `$ Date :` header block;
* `NASTRAN ...` / `INIT ...` statements ahead of the executive control — carried into every check
  deck so the check runs use the same solver system settings as the user's own runs.

The case-control commands that change the model itself — `NSM`, `MPC`, DMIG selections — are also
carried (`model.CARRIED_CASE_CONTROL`). A 1G template from this model selected `NSM = 2`; dropping
it would silently drop mass.

In [3]:
for k, v in info.trace.items():
    print(f"{k:20s} {v}")
print("carried case control:", info.carried_case_control or "none")

deck                 /home/user/FEMAP_API/1G_unit_load_check-0000.dat
nastran_statements   ['INIT MASTER(S)', 'NASTRAN SYSTEM(442)=-1,SYSTEM(319)=1']
title                1G UNIT LOAD CHECK
subtitle             BD IRAD V2
femap_version        2512.0
femap_model          D:\IRAD\BD_IRAD_FEMv4.modfem
femap_date           Mon Nov  2 13:49:56 2026
carried case control: none


### 3.2 Element quality

Nastran has its own element-geometry limits: `GEOMCHECK` (qrg ~8150, remark 2), applied on every run
with defaults such as CQUAD4 interior angle 30°–150°, edge aspect ratio 100, CTRIA3 maximum angle
160°. Nastran only prints informational messages for elements beyond them (solid aspect-ratio checks
are FATAL by default). FEMAP's *Tools > Check > Distortion* uses its own user-set tolerances.

The tool computes element quality with pyNastran and counts elements beyond the **Simcenter
defaults** for the tests whose definition matches pyNastran's: interior angles and aspect ratio.
Nastran's skew, taper and warp are defined differently from pyNastran's metrics, so those are
statistics only. Quality is **INFO** — it never fails the run.

In [4]:
print(f"{'test':10s} {'types':16s} {'limit':8s} {'checked':>8s} {'beyond':>7s}  worst")
for row in info.geomcheck:
    print(f"{row[0]:10s} {row[1]:16s} {row[2]:8s} {row[3]:8d} {row[4]:7d}  {row[5]}")
for name, st in info.quality.items():
    print(f"{name:20s} min {st['min']:9.4g}  mean {st['mean']:9.4g}  max {st['max']:9.4g}")

test       types            limit     checked  beyond  worst
Q4_IAMIN   CQUAD4/CQUADR    < 30         7620       0  -
Q4_IAMAX   CQUAD4/CQUADR    > 150        7620       0  -
QAD_AR     CQUAD4           > 100        7620       0  -
T3_IAMAX   CTRIA3/CTRIAR    > 160           8       0  -
min_interior_angle   min     6.956  mean     62.32  max        90
max_interior_angle   min     77.69  mean     94.86  max     143.8
dideal_theta         min         0  mean     5.097  max     53.84
max_skew_angle       min         0  mean     2.817  max     53.95
max_warp_angle       min         0  mean 7.166e-05  max   0.01638
max_aspect_ratio     min     1.004  mean     1.271  max     2.924
area_ratio           min         1  mean     1.018  max     1.908
taper_ratio          min         0  mean   0.05025  max     1.878
min_edge_length      min      0.05  mean    0.5012  max     1.358


## 4. Choose the kinematic mount grid

Unit-displacement, thermal, GROUNDCHECK and WEIGHTCHECK all need one reference grid. FEMCI wants it
near the CG (femci ~1966). `model.choose_mount` accepts a grid only if:

1. it has **rotational stiffness** — it is on a shell, beam, bar or bush, or is an RBE2
   independent grid. A grid on solids only has no rotational DOF stiffness, so the R1–R3
   unit-rotation subcases would "pass" without testing anything;
2. it is **not dependent** (m-set) on a rigid element or MPC — Nastran cannot SPC an m-set DOF;
3. it has **no permanent constraint** (PS field on its GRID card). SPC cards in the model's own
   sets don't matter: the check decks select only their own SPC set;
4. its output system is **CD = 0**, so SPCD DOF 1–6 are basic X/Y/Z.

With `mount_grid = 0` the qualifying grid nearest the CG is chosen. On this model that is grid
300000 (the ball-assembly RBE2 independent grid, 4.45 from the CG); the user's own runs used
199999 (16.6 from the CG), which also qualifies.

In [5]:
auto = M.choose_mount(model, info, 0)
print(f"auto-pick: grid {auto.grid} at {auto.xyz}, {auto.distance_to_cg:.4g} from CG")
mount = M.choose_mount(model, info, s.mount_grid)
print(f"used:      grid {mount.grid} at {mount.xyz}, {mount.distance_to_cg:.4g} from CG; "
      f"{len(mount.rotation_grids)} grids have rotational stiffness")
try:
    M.choose_mount(model, info, 200676)     # a dependent grid of RBE2 110225
except ValueError as exc:
    print("refused:", exc)

auto-pick: grid 300000 at [0.0, 0.0, 0.0], 4.453 from CG
used:      grid 199999 at [0.0, -20.988, 0.0], 16.56 from CG; 7302 grids have rotational stiffness
refused: Mount grid rejected: grid 200676 is a dependent (m-set) grid of a rigid element/MPC (components 123456); Nastran cannot SPC it


## 5. Write the bulk data once

`model.bulk_text` writes every bulk card through pyNastran in **16-character large-field** format so
no coordinate or property loses precision. It drops only the PARAMs each check deck sets for itself
(AUTOSPC, GRDPNT, POST, OGEOM, PRGPST). K6ROT, WTMASS and every other PARAM stay, because they change
the answers.

The cell below proves the rewrite is lossless: it reads `model_bulk.bdf` back and compares every card
field-by-field with the original deck.

In [6]:
(out / "model_bulk.bdf").write_text(M.bulk_text(model))

from pyNastran.bdf.bdf import BDF
(out / "roundtrip.dat").write_text("SOL 101\nCEND\nBEGIN BULK\nINCLUDE 'model_bulk.bdf'\nENDDATA\n")
back = BDF(debug=False, mode="nx"); back.read_bdf(str(out / "roundtrip.dat"), xref=False)
n = bad = 0
for name in ("nodes", "elements", "properties", "materials", "rigid_elements", "masses", "coords"):
    a, b = getattr(model, name), getattr(back, name)
    for k in a:
        n += 1
        bad += a[k].repr_fields() != b[k].repr_fields()
print(f"{n} cards compared, {bad} differ;  PARAMs kept: {sorted(back.params)}")
(out / "roundtrip.dat").unlink()

41538 cards compared, 0 differ;  PARAMs kept: ['K6ROT']


### 5.1 The thermal copy

The thermal soak needs one CTE everywhere (femci ~2319). `model.make_thermal_uniform` works on a
**fresh read** of the deck, never on the model used for the other checks:

* every MAT1/2/3/8/9/11 gets $\alpha$ (shear CTE terms 0) and TREF 0; PCOMP/PCOMPG TREF 0;
* MATT* entries are removed — with `TEMP(INIT)` defined, MATTi properties are evaluated at the
  initial temperature (qrg ~29940 remark 8), which could make $\alpha$ non-uniform;
* every rigid element gets `ALPHA` = $\alpha$ (the deck adds `RIGID = LAGRAN`, without which rigid
  elements get no thermal load at all).

In [7]:
tmodel = M.read_model(Path(s.deck), out / "work")
thermal_log = M.make_thermal_uniform(tmodel, s.alpha)
(out / "model_bulk_thermal.bdf").write_text(M.bulk_text(tmodel))
print("\n".join(thermal_log))

uniform CTE = 1e-05, TREF = 0 set on: 4 MAT1, 16 RBE2, 2 RBE3


## 6. Write the check decks

`decks.write_decks` writes five decks. Each is a provenance header, the carried NASTRAN statements,
executive + case control, a few bulk cards of its own, and `INCLUDE 'model_bulk.bdf'`.

Added cards use a reserved ID block (`model.reserved_base` finds a block of 1000 IDs, starting at
9,000,000, that collides with nothing in the model). The kinematic mount is the card
`SPC1, <base+1>, 123456, <mount grid>`: `base+1` is only the **set ID** the case control selects,
and the constrained grid is always the chosen mount grid.

Case-control syntax is from the QRG and from the user's own working decks: `GROUNDCHECK` (qrg ~18934),
`WEIGHTCHECK ... MASS` (qrg ~31138; MASS selects mass rather than weight units), `MEFFMASS`
(qrg ~22311), `SPCD` (qrg ~119845, its DOF must also be on a selected SPC1), `TEMPERATURE(INITIAL)`
and `(LOAD)` (qrg ~29875), `RIGID = LAGRAN` (qrg ~26012).

In [8]:
base = M.reserved_base(model)
ids = D.DeckIds(base)
gravity_spc = ids.mount_spc if s.gravity_mount == "kinematic" else s.real_spc_set
which = [k for k in D.DECK_NAMES if s.checks.get(k)]
paths = D.write_decks(out, which, ids, mount.grid, gravity_spc, s.real_spc_set, UNIT_SYSTEMS[s.units]["g"],
                      s.n_modes_freefree, s.n_modes_constrained, s.autospc_freefree,
                      info.carried_case_control, info.trace)
print("reserved ID base:", base)
print(paths["unit_disp"].read_text())

reserved ID base: 9000000
$ Written by fem_checks (FEM validation). Regenerate instead of editing.
$ Written    : 2026-11-03 19:01
$ Source deck: /home/user/FEMAP_API/1G_unit_load_check-0000.dat
$ FEMAP model: D:\IRAD\BD_IRAD_FEMv4.modfem  (exported Mon Nov  2 13:49:56 2026)
$ Model title: 1G UNIT LOAD CHECK / BD IRAD V2
INIT MASTER(S)
NASTRAN SYSTEM(442)=-1,SYSTEM(319)=1
SOL 101
CEND
  TITLE = 1G UNIT LOAD CHECK
  SUBTITLE = BD IRAD V2
  LABEL = FEM CHECK 03 UNIT ENFORCED DISPLACEMENT
  ECHO = NONE
  SPC = 9000001
  DISPLACEMENT(PLOT) = ALL
  SPCFORCE(PLOT) = ALL
  OLOAD(PLOT) = ALL
  FORCE(PLOT,CORNER) = ALL
SUBCASE 1
    LABEL = FEM CHECK 03 UNIT ENFORCED T1 AT GRID 199999
    LOAD = 9000011
SUBCASE 2
    LABEL = FEM CHECK 03 UNIT ENFORCED T2 AT GRID 199999
    LOAD = 9000012
SUBCASE 3
    LABEL = FEM CHECK 03 UNIT ENFORCED T3 AT GRID 199999
    LOAD = 9000013
SUBCASE 4
    LABEL = FEM CHECK 03 UNIT ENFORCED R1 AT GRID 199999
    LOAD = 9000014
SUBCASE 5
    LABEL = FEM CHECK 03 UNI

## 7. Run Nastran (or reuse the sample outputs)

`runner.run_nastran` calls `nastran.exe <deck> scr=yes old=no batch=no` in the output folder and waits.
Before each run it deletes that deck's old `.f06/.op2/.f04/.log`, so a failed run can never be parsed
as if it were the new one. A run with no `.f06`, or with a FATAL message, makes that check **ERROR**
(never FAIL — the model wasn't judged).

In `DEMO` mode the user's real sample outputs are copied in instead. There are samples for the
free-free, unit-displacement and gravity runs; the constrained-modes and thermal checks will show
ERROR here because no sample exists for them.

In [9]:
SAMPLES = {"freefree": "freefree_check-0000", "unit_disp": "ud_check-0000",
           "gravity": "1g_unit_load_check-0000"}
runs = {}
for key, deck in paths.items():
    if DEMO:
        if key in SAMPLES:
            for suf in (".f06", ".op2"):
                src = REPO / (SAMPLES[key] + suf)
                if src.exists():
                    shutil.copy(src, deck.with_suffix(suf))
        f06 = deck.with_suffix(".f06"); op2 = deck.with_suffix(".op2")
        runs[key] = RunResult(str(deck), str(f06) if f06.exists() else None,
                              str(op2) if op2.exists() else None, None, 0.0,
                              None if f06.exists() else "no sample output")
    else:
        runs[key] = run_nastran(s.nastran_exe, deck, s.nastran_args, s.run_timeout_s)
    print(f"{deck.name:28s} f06={'yes' if runs[key].f06 else 'no ':3s} op2={'yes' if runs[key].op2 else 'no'}")

01_freefree.dat              f06=yes op2=no
02_constrained_modes.dat     f06=no  op2=no


03_unit_disp.dat             f06=yes op2=yes


04_gravity.dat               f06=yes op2=yes
05_thermal.dat               f06=no  op2=no


## 8. Parse and evaluate

### 8.0 The `.f06` parsers

pyNastran reads the OP2 binary results, but several tables only exist in the `.f06`: GROUNDCHECK,
WEIGHTCHECK, the OLOAD/SPCFORCE resultants, EPSILON, the AUTOSPC singularity count and the
FATAL/WARNING messages. `f06.py` parses these by their headings (never by version banner — the
samples come from two Simcenter versions) and survives page breaks.

In [10]:
f06s = {k: parse_f06(r.f06) if r.f06 else None for k, r in runs.items()}
ff = f06s["freefree"]
print("version", ff.version, "| completed", ff.completed)
for g in ff.groundcheck:
    print(f"GROUNDCHECK {g.set_name:10s} limit {g.limit:9.4g}  max SE {max(g.strain_energy):9.3e}  {'PASS' if g.all_pass else 'FAIL'}")
print("first 8 modes (Hz):", [round(m.cycles, 6) for m in ff.modes[:8]])
print("AUTOSPC singularities:", ff.singularities, "| messages:", [(m.kind, m.number) for m in ff.messages])

version 2512.0095 | completed True
GROUNDCHECK G          limit     2.337  max SE 1.209e-04  PASS
GROUNDCHECK N          limit     11.78  max SE 1.026e-03  PASS
GROUNDCHECK N+AUTOSPC  limit     11.78  max SE 1.026e-03  PASS
GROUNDCHECK F          limit     11.78  max SE 1.026e-03  PASS
GROUNDCHECK A          limit     11.78  max SE 1.026e-03  PASS
first 8 modes (Hz): [0.000429, 0.000411, 6.9e-05, 5.3e-05, 0.000104, 0.00025, 133.0038, 180.3932]
AUTOSPC singularities: [(51879, 51879)] | messages: [('WARNING', '2125')]


### 8.1 Free-free modes + GROUNDCHECK (femci ~2082)

*Physics:* with no constraints the model must have exactly six zero-energy (rigid-body) modes. A
seventh near-zero mode is a mechanism; rigid-body modes that are not near zero mean something is
grounding the model.

*Criterion:* $\dfrac{\max(f_1,\dots,f_6)}{f_7} \le 10^{-4}$ — six decades between the last rigid mode
and the first elastic one. A mechanism makes $f_7\approx0$ and fails it. GROUNDCHECK moves the model
rigidly through each DOF set (G, N, N+AUTOSPC, F, A) and prints the strain energy against Nastran's
own limit (largest stiffness term / 1e10, qrg ~18934); all sets must PASS. The WEIGHTCHECK % change
per set shows mass lost or gained by MPCs/RBEs during reduction (INFO).

In [11]:
r01 = C.check_freefree(ff, s.thresholds, s.autospc_freefree)
print(r01.status, "-", r01.summary)
for d in r01.details: print("  *", d)

PASS - max rigid-body 4.29e-04 Hz, first elastic 133 Hz, ratio 3.2e-06; GROUNDCHECK all sets PASS
  * Criterion: max(f1..f6) / f7 <= 1e-04 (decision #1). Result 3.22e-06.
  * FEMCI's absolute 1e-4 Hz limit (femci ~2095) is deliberately not applied: it doesn't scale with model size or units, and the 6th/7th separation is what shows grounding.
  * GROUNDCHECK passes only if every requested set (G, N, N+AUTOSPC, F, A) prints PASS in all 6 directions.
  * Free-free run used PARAM,AUTOSPC,YES. FEMCI (~2127): AUTOSPC can make the A-set fail without real grounding.
  * WEIGHTCHECK: 0% rigid-body mass change in every reduced set.


### 8.2 Constrained modes

Run on the model's real boundary-condition set with MEFFMASS. **INFO only** (user decision): gimbals
legitimately have one or two near-zero bearing modes depending on whether the bearing rotations are
locked, so any frequency floor would fail good models. The report shows frequencies and the
effective-mass fraction per mode and cumulative.

In [12]:
r02 = C.check_constrained(f06s.get("constrained_modes"), f"model SPC set {s.real_spc_set}")
print(r02.status, "-", r02.summary)

ERROR - Nastran did not produce an .f06 for this check.


### 8.3 Unit enforced displacement and rotation (femci ~1953)

*Physics:* hold one grid in all six DOFs and enforce 1.0 in one DOF at a time. A correctly built
model moves as a rigid body, with no reaction and no internal force.

*Three criteria per subcase, all must pass:*

1. **Mount reaction** — the SPCFORCE resultant, moved from the basic origin to the mount
   ($\mathbf M_m = \mathbf M_0 - \mathbf r_m\times\mathbf F$), within 0.1 lbf and 1.0 in-lbf
   (femci ~1965, converted to the model's units).
2. **Displacement field** — every grid's displacement (transformed from its CD frame to basic)
   against the rigid-body field. Translation $k$: $\mathbf u=\mathbf e_k$. Rotation $k$:
   $\mathbf u=\mathbf e_k\times(\mathbf r-\mathbf r_m)$, $\boldsymbol\theta=\mathbf e_k$ — exact in
   linear statics. Error $=\max\lVert\mathbf u-\mathbf u_{rb}\rVert/\max\lVert\mathbf u_{rb}\rVert\le10^{-3}$.
   Rotations are compared only on grids with rotational stiffness.
3. **Element forces** at element **centroids** within the same limits (shell forces are per unit
   length).

*Why centroids:* in the sample run the CQUAD4 CORNER (bilinear-extrapolated) values reach 33–82
lbf/in in about 88 elements, only in the three rotation subcases, while the same elements' centroid
values are ~1e-7. That is a recovery artifact under a 1 rad rigid rotation, not grounding. Corner
peaks above the limit are still listed (INFO) with their element IDs.

In [13]:
op2 = R.load_op2(runs["unit_disp"].op2)
fields = R.displacements_basic(op2, model, {})
peaks = {sc: R.element_force_peaks(op2, sc) for sc in range(1, 7)}
r03 = C.check_unit_disp(f06s["unit_disp"], fields, peaks, mount.xyz, set(mount.rotation_grids),
                        s.thresholds, s.force_tol(), s.moment_tol(), UNIT_SYSTEMS[s.units])
print(r03.status, "-", r03.summary)
for t in r03.tables[:2]:
    print("\n" + t.title); print("  ", t.headers)
    for row in t.rows: print("  ", row)

PASS - 6 subcases rigid: reactions, displacement field (<= 1e-03) and element forces within limits

Mount reaction (SPCFORCE resultant moved to the mount)
   ['Subcase', '|F| (lbf)', '|M| (in-lbf)', 'Result']
   ['T1', '5.015e-06', '3.997e-06', 'ok']
   ['T2', '4.572e-06', '1.010e-05', 'ok']
   ['T3', '6.326e-06', '6.224e-06', 'ok']
   ['R1', '4.827e-06', '2.785e-04', 'ok']
   ['R2', '5.842e-06', '1.786e-04', 'ok']
   ['R3', '1.014e-05', '9.700e-05', 'ok']

Displacement field vs rigid-body motion
   ['Subcase', 'Translation err', 'Rotation err', 'Rot. grids', 'Result']
   ['T1', '6.889e-11', '4.638e-12', 7302, 'ok']
   ['T2', '8.090e-11', '1.307e-11', 7302, 'ok']
   ['T3', '6.925e-11', '4.647e-12', 7302, 'ok']
   ['R1', '2.998e-08', '5.526e-12', 7302, 'ok']
   ['R2', '3.878e-08', '5.742e-12', 7302, 'ok']
   ['R3', '3.809e-08', '6.575e-12', 7302, 'ok']


### 8.4 Unit gravity X / Y / Z (femci ~2231)

*Physics:* under 1 g the applied load (OLOAD) must equal the model's weight, and the reactions
(SPCFORCE) must balance it exactly. Anything grounded outside the selected mount — a spring to
ground, an AUTOSPC'd DOF that carries load — takes load the reactions don't see.

*Criteria, per subcase, all ≤ 1e-4:*

* force balance $\lVert\mathbf F_{OL}+\mathbf F_{SPC}\rVert/\lVert\mathbf F_{OL}\rVert$
* moment balance $\lVert\mathbf M_{OL}+\mathbf M_{SPC}\rVert/(\lVert\mathbf M_{OL}\rVert+\lVert\mathbf F_{OL}\rVert L)$
* weight $\bigl|\,\lVert\mathbf F_{OL}\rVert-m g\bigr|/(m g)$, with $m$ from WEIGHTCHECK (MASS units)

Vector norms, not per-component percentages: the off-axis components are round-off noise
(e.g. SPC $10^{-8}$ against an OLOAD of exactly 0) that a per-component test would fail on.

*Note for this demo:* the sample gravity run used $g$ = 386.1; the tool writes 386.0886. The
weight comparison below therefore uses the sample's own $g$ so it compares like with like.

In [14]:
fg = f06s["gravity"]
mass = next(b.mass[0] for b in fg.mass_blocks if b.mass)
r04 = C.check_gravity(fg, mass, 386.1 if DEMO else UNIT_SYSTEMS[s.units]["g"], info.size,
                      s.thresholds, UNIT_SYSTEMS[s.units], "kinematic mount")
print(r04.status, "-", r04.summary)
for row in r04.tables[0].rows: print("  ", row)

PASS - OLOAD = -SPCFORCE and = mass*g to <= 1e-04 in X, Y, Z (kinematic mount)
   ['X', '733.704', '-733.704', '733.704', '6.886e-11', '2.801e-11', '1.356e-07', 'ok']
   ['Y', '733.704', '-733.704', '733.704', '1.003e-10', '7.816e-12', '1.356e-07', 'ok']
   ['Z', '733.704', '-733.704', '733.704', '9.036e-11', '1.279e-11', '1.356e-07', 'ok']


### 8.5 Thermal soak, +1° (femci ~2276)

*Physics:* with one CTE everywhere, a uniform temperature change on a kinematic mount is
**exactly** stress-free for any mix of elements, and every grid moves
$\mathbf u = \alpha\,\Delta T\,(\mathbf r-\mathbf r_m)$ with no rotation. This replaces the Book's
THDEF program (a legacy GSFC VAX tool, femci ~2285).

*Criteria:* displacement error ≤ 1e-3 relative to the theory field; peak element stress
$\sigma_{max}/(E_{max}\,\alpha\,\Delta T)\le10^{-3}$ ($E\alpha\Delta T$ is the fully-restrained
thermal stress, the natural scale). The 20 most-stressed elements are listed with the springs,
bushes and rigid elements they touch — the usual culprits.

$\Delta T = 1$ comes from `TEMPERATURE(INITIAL)` = 0 and `TEMPERATURE(LOAD)` = 1; with INITIAL
defined, Simcenter computes $\varepsilon=\alpha(T_{load}-T_{init})$ regardless of TREF
(qrg ~29940). No sample thermal run exists, so this shows ERROR in demo mode.

In [15]:
r05 = C.check_thermal(f06s.get("thermal"), None, [], mount.xyz, s.alpha, info.e_max, s.thresholds,
                      {}, model, UNIT_SYSTEMS[s.units], [])
print(r05.status, "-", r05.summary)

ERROR - Nastran did not produce an .f06 for this check.


### 8.6 Mass, CG and inertia vs CAD targets

Optional: compared only for the values the user enters. FE values come from the free-free
WEIGHTCHECK G-set (mass units, `MASS` option): mass, CG, and the inertia matrix $I(S)$ about the CG.

**Sign of the products of inertia.** In the sample `.f06`, $I(S)_{xy}=+5.385$ while the rigid-body
mass matrix gives $MO_{45}=-2.049$ about the origin. Shifting to the CG,
$-2.049 + m\,x\,y = -2.049 + 1.9003(0.39607)(-4.43269) = -5.385$, so $MO$ holds the inertia
*tensor* ($-\int xy\,dm$) and $I(S)$ prints $+\int xy\,dm$ — the CAD convention. Targets entered as
"tensor" are sign-flipped before comparing. Product tolerances are judged as a % of the largest
principal moment, because products are often near zero.

In [16]:
gblock = next(b for b in ff.mass_blocks if b.source == "WEIGHTCHECK" and b.dof_set == "G")
print("FE:", C.mass_from_block(gblock))
r06 = C.check_mass_props(gblock, s.targets, s.units, UNIT_SYSTEMS[s.units])
print(r06.status, "-", r06.summary)

FE: {'mass': 1.900295, 'cg': [0.3960659, -4.432694, 7.212851e-06], 'inertia': [460.2007, 389.1584, 387.3969, 5.385219, 2.4768e-05, -0.0002109791]}
PASS - 1 supplied value(s) within tolerance


## 9. Report

`report.build_pdf` writes a landscape PDF: page 1 has the model, traceability and the PASS/FAIL table;
then one section per check with its criteria, tables and chart; then an appendix with run times and
every FATAL/WARNING message. `results.json` holds every number for comparing runs.

Chart colors are the first three slots of a palette validated for color-vision deficiency; every
chart's numbers are also in a table, so nothing depends on color alone.

In [17]:
r00 = C.check_model_summary(info, mount, base, "kinematic mount", f"model SPC set {s.real_spc_set}")
results = [r00, r01, r02, r03, r04, r05, r06]
charts_dir = out / "charts"; charts_dir.mkdir(exist_ok=True)
charts = {"freefree": RP.chart_freefree(r01.data, charts_dir / "freefree.png", s.thresholds.rbm_ratio)}
q = RP.chart_quality(info.quality, charts_dir / "quality.png")
if q: charts["model_summary"] = q
header = {"Model deck": s.deck, "Model title / subtitle": f"{info.trace.get('title')} / {info.trace.get('subtitle')}",
          "FEMAP model": info.trace.get("femap_model", "-"), "Units": s.units,
          "Mount grid": f"{mount.grid}", "Mode": "DEMO (sample outputs)" if DEMO else "live Nastran"}
messages = [(Path(r.deck).name, m) for k, r in runs.items() if f06s.get(k) for m in f06s[k].messages]
pdf = RP.build_pdf(out / "fem_checks_report.pdf", header, results, charts, messages, list(runs.values()))
for r in results: print(f"{r.status:8s} {r.title}")
print("report:", pdf)

INFO     Model summary
PASS     Free-free modes + GROUNDCHECK
ERROR    Constrained modes
PASS     Unit enforced displacement / rotation
PASS     Unit gravity (OLOAD vs SPCFORCE)
ERROR    Thermal soak (+1 deg, uniform CTE, kinematic mount)
PASS     Mass, CG and inertia vs targets
report: /tmp/fem_checks_demo/fem_checks_report.pdf


## 10. Reading the result, and what to do on a FAIL

| Result | Usual causes | Where to look |
|---|---|---|
| Free-free ratio FAIL | grounded spring (CELAS/CBUSH to one grid), RBE/MPC that doesn't allow rigid motion, missing connection (7th mode ≈ 0) | GROUNDCHECK set that fails first: G → springs/DMIG, N → MPC/RBE3, F → SPCs, A → AUTOSPC (femci ~2110) |
| Unit-disp reaction or field FAIL | something besides the mount is grounded | worst-grid list in the report |
| Unit-disp element force FAIL | CBUSH/CELAS between non-coincident grids, offsets | worst-element list |
| Gravity FAIL | load leaking to AUTOSPC'd DOFs or grounded elements; weight error = mass/NSM/WTMASS problem | resultant table, mass*g column |
| Thermal FAIL | spring/bush connections, anisotropic materials the overwrite missed, rigid elements without LAGRAN | top-20 element list with "touches" column |
| ERROR | Nastran FATAL or missing output — the model wasn't judged | appendix messages, the run's `.f04/.log` |

## 11. Limits

* Simcenter Nastran only; parsers tested on Simcenter 2306 and 2512 output.
* pyNastran 1.4.1 has no RTRPLT card; a deck with one stops at read time.
* Mount grid must have CD = 0; mass targets must be about the CG in the FEM basic frame.
* pyNastran's CG (used only for the auto-pick) ignores NSM and CONM2 offsets in a non-basic CID; the
  GPWG values in the report are authoritative.
* Nothing in this repo can run Nastran: the decks themselves are verified by the first live run
  (checklist in `plan-fem-checks.md`).